# Упражнения к семинару 12. Гипотеза о среднем: одна выборка и доверительные интервалы

Переименуйте файл в формате `Группа-Фамилия-Имя-seminar-12-tasks.ipynb`, например `2MP9-Ivanov-Ivan-seminar-12-tasks.ipynb`: этот файл сдаётся в Moodle. Формулировки упражнений здесь те же, что в листке `seminar-12-tasks.md`; дополнительно в ноутбуке заготовки функций, тесты и ячейки «Проверка».

Данные: таблицы Московской области за 2022–2025 годы, ДТП `datasets/legal/dtp-moscow-region.csv` и участники `datasets/legal/dtp-moscow-region-participants.csv`; путь в ячейках задан от папки `exercises/`, поэтому ноутбук должен лежать в ней, а папка `datasets` рядом, как в репозитории.

- Часть А: выполните ячейку с подключением библиотек, соберите таблицу водителей в упражнении А1, затем решайте упражнения по порядку и вводите числа и решения в тест Moodle «Семинар 12».
- Части Б и В: напишите тело функции вместо `pass`, запустите ячейку с тестами; если всё верно, она напечатает `Все тесты прошли успешно!`. Затем выполните ячейку «Проверка»: она вызывает функцию на таблицах области и печатает числа, которые вводятся в тест вместе с кодом функции.

Правила: функция не ссылается на переменные, определённые вне неё, кроме `pd`, `np`, `plt`, `sns`, `stats`, `proportions_ztest`, `proportion_confint` и функций части Б; ячейки с тестами не меняются; в коде решений библиотеки, кроме pandas, NumPy, matplotlib, seaborn, scipy и statsmodels, не используются.

## Часть А. Разминка

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

sns.set_theme(style='whitegrid')       # оформление графиков по умолчанию
pd.set_option('display.width', 200)   # широкие таблицы печатаются без переноса

### А1. Средний стаж против 15 и 18 лет

Загрузите таблицу ДТП `datasets/legal/dtp-moscow-region.csv` в `df` с `index_col='id'` и `parse_dates=['datetime']` и таблицу участников `datasets/legal/dtp-moscow-region-participants.csv` в `people`. Соберите таблицу водителей с их ДТП, как в разделе 1 ноутбука занятия: стаж без кодов 95, 96 и 99 в новом столбце `people['experience_clean']`; `drivers = people[people['role'] == 'Водитель'].merge(df, left_on='dtp_id', right_index=True)` — должно получиться 25 364 водителя; возраст машины `drivers['vehicle_age']` — год ДТП минус `vehicle_year`, значения больше 100 — пропуск через `where`. Стаж без пропусков — `experience = drivers['experience_clean'].dropna()`. Проверьте t-тестом одной выборки две гипотезы с двусторонней альтернативой: средний стаж водителей, попавших в ДТП, равен 15 годам; равен 18 годам. Посчитайте 95-процентный доверительный интервал для среднего стажа. Какова t-статистика против 15, до десятых, и каково решение при 0,05? Каково p против 18, до тысячных, и каково решение? Каковы границы интервала, до сотых? Подсказка: разделы 1 и 2 ноутбука занятия; `stats.ttest_1samp(experience, 15)`, поля результата — `statistic` и `pvalue`; интервал — `stats.t.interval(0.95, len(experience) - 1, loc=experience.mean(), scale=stats.sem(experience))`. Решения согласуются с интервалом: значение внутри интервала тест не отвергает, снаружи — отвергает. По столбцу `drivers['experience_clean']` с пропусками тест возвращает `nan`, а сравнение `nan < 0.05` молча даёт «не отвергаем»; стаж с кодами, столбец `experience`, даёт другое среднее.

In [ ]:
# А1

### А2. Городской округ как выборка

Водители городского округа Домодедово — выборка, все водители области — генеральная совокупность со средним стажем `experience.mean()`. Стаж водителей Домодедово без пропусков: `district = drivers.loc[drivers['district'] == 'Домодедово', 'experience_clean'].dropna()`. Сколько в нём значений и каково среднее, до сотых? Каков 95-процентный доверительный интервал, до сотых? Каково p двустороннего t-теста против среднего области, до тысячных, и каково решение при 0,05? Подсказка: раздел 3 ноутбука занятия; проверяемое значение — число `experience.mean()`, а не 15 и не 18. Интервал Домодедово шире интервала всей области из А1: значений меньше, и ширина растёт как единица, делённая на корень из n. Среднее области лежит внутри интервала — поэтому тест и не отвергает; но «не отвергаем» не значит, что стаж в Домодедово такой же, как в области: интервал допускает стаж и почти на год меньше областного.

In [ ]:
# А2

### А3. Односторонняя гипотеза

Вопрос поставлен до данных: у водителей Сергиево-Посадского района, `drivers['district'] == 'Сергиево-Посадский'`, стаж больше, чем в среднем по области? Нулевая гипотеза: среднее района равно среднему области `experience.mean()`; альтернативная — больше. Выполните t-тест против среднего области с двусторонней альтернативой и с альтернативой `alternative='greater'`. Каковы оба p, до тысячных, и каковы решения при 0,05? Подсказка: раздел 3 ноутбука занятия; значения параметра `alternative` у scipy — `'two-sided'`, `'less'`, `'greater'`. Если среднее отклонилось в заявленную сторону, односторонний p вдвое меньше двустороннего. `alternative='less'` даёт p, близкое к единице: направление перепутано. Односторонний тест честен, только если направление выбрано до взгляда на данные, как здесь — в условии. Водителей в районе меньше 5 % водителей области, поэтому среднее области годится для него как известное значение.

In [ ]:
# А3

### А4. Доля ДТП с погибшими

ДТП с погибшими — ДТП, где `dead_count` больше нуля; доля считается по таблице `df`: одно ДТП — одна строка. Проверьте z-тестом для доли гипотезу «доля ДТП с погибшими равна 0,15» с двусторонней альтернативой и посчитайте 95-процентный интервал Вилсона. Каковы z, до сотых, и p, до тысячных, и каково решение при 0,05? Каковы границы интервала, до тысячных? Подсказка: раздел 5 ноутбука занятия; `proportions_ztest(count, nobs, 0.15, prop_var=0.15)` — сначала число ДТП с погибшими, потом число всех ДТП, потом проверяемая доля, и она же в `prop_var`; результат — кортеж `(z, p)`; интервал — `proportion_confint(count, nobs, method='wilson')`. Без `prop_var` получится тест Вальда и другая z. С аргументами в обратном порядке statsmodels молча возвращает огромную z и p 0. Доля погибших среди участников, по таблице `people`, — другой вопрос и другое число.

In [ ]:
# А4

### А5. Доля ДТП по выходным

Если день недели не влияет на ДТП, на субботу и воскресенье приходится 2/7 всех ДТП. Посчитайте долю ДТП по выходным, `df['datetime'].dt.dayofweek >= 5`, и проверьте z-тестом для доли гипотезу «доля равна 2/7» с двусторонней альтернативой. Какова доля, до десятитысячных? Каковы z, до сотых, и p, до тысячных, и каково решение при 0,05? Подсказка: раздел 5 ноутбука занятия; номера дней `dt.dayofweek`: 0 — понедельник, 5 — суббота, 6 — воскресенье; проверяемая доля — `2 / 7`, а не округлённое 0,29, и она же в `prop_var`. `dayofweek > 5` берёт одно воскресенье. Почему решение именно такое, покажет А6: какие дни недели перегружены.

In [ ]:
# А5

### А6. Согласие с равномерным распределением

ДТП по дням недели: `counts = df['datetime'].dt.dayofweek.value_counts().sort_index()`. Проверьте хи-квадратом согласия гипотезу «ДТП распределены по дням недели поровну»: ожидаемые — `counts.sum() / 7` ДТП на каждый день. Посчитайте вклад каждого дня в статистику, `(counts - expected) ** 2 / expected`, и нарисуйте столбцы ДТП по дням с линией ожидаемого, как в разделе 6 ноутбука занятия. Какова статистика χ², до десятых, и каково решение при 0,05? Какой день недели даёт наибольший вклад? Подсказка: раздел 6 ноутбука занятия; `stats.chisquare(counts, expected)`; номер дня с наибольшим вкладом — `idxmax()` у Series вкладов, 0 — понедельник, 6 — воскресенье. Ожидаемые, заданные долями `[1 / 7] * 7`, дают `ValueError`: суммы наблюдаемых и ожидаемых не сходятся. Без `sort_index()` статистика та же, но `argmax()` вернёт позицию в ряду, а не номер дня. Сравните с А5: перегруженный день один, а доля выходных складывается из двух дней.

In [ ]:
# А6

### А7. Много районов

Для каждого района или городского округа с 50 и более ДТП проверьте z-тестом для доли гипотезу «доля ДТП по выходным такая же, как во всей области»: число ДТП в субботу и воскресенье и число ДТП района против доли выходных по всей таблице `df`. Сколько районов проверено? У скольких p меньше 0,05? У скольких p меньше порога с поправкой Бонферрони — 0,05, делённого на число проверенных районов? Подсказка: раздел 7 ноутбука занятия; `df.assign(weekend=df['datetime'].dt.dayofweek >= 5).groupby('district')['weekend'].agg(['sum', 'count'])`, затем отбор `count >= 50` и p списком по парам `zip(…['sum'], …['count'])`: `proportions_ztest(k, n, p0, prop_var=p0)`, где p0 — доля выходных по всей таблице. Отбор `> 50` теряет округ ровно с 50 ДТП; поправка делит 0,05 на число проверенных районов, а не всех значений `district`; без `prop_var` получается тест Вальда и другие числа значимых; районы, посчитанные по таблице водителей, — другие числа: ДТП с несколькими машинами в ней повторяются.

In [ ]:
# А7

## Часть Б. Основные задачи

### Б1. t-тест одной выборки

Первый инструмент для гипотез этапа 3 о среднем: средний стаж против нормы, средний срок рассмотрения против регламента, район против области. Тест — `stats.ttest_1samp`; пропуски убираются заранее, иначе scipy вернёт `nan`. Альтернатива — как у scipy: `'two-sided'` — среднее не равно mu, `'less'` — меньше, `'greater'` — больше; её выбирают до взгляда на данные.

Реализуйте функцию `t_one(values, mu, alternative='two-sided')`, которая проверяет t-тестом одной выборки гипотезу о том, что среднее равно mu. Аргументы: `values` — столбец чисел, Series; пропуски допускаются; `mu` — проверяемое среднее, число; `alternative` — альтернативная гипотеза, строка 'two-sided', 'less' или 'greater'; по умолчанию 'two-sided'. Возвращаемое значение: кортеж из двух чисел (t, p): статистика и p-значение stats.ttest_1samp по значениям values без пропусков против mu с альтернативой alternative. Series не меняется.

Подсказка: раздел 1 ноутбука занятия: `result = stats.ttest_1samp(values.dropna(), mu, alternative=alternative)`, `return result.statistic, result.pvalue`. t по формуле со стандартным отклонением `np.std` делит на n, а не на n − 1; z-тест по нормальному распределению вместо распределения Стьюдента на маленькой выборке даёт заниженное p.

Проверка в ноутбуке: возраст машины водителей области против 10,5 года через t_one, с двусторонней альтернативой и с альтернативой «больше»; печатаются t и p обеих проверок, в тест вводятся t до сотых, оба p до тысячных и решение двусторонней проверки.

In [ ]:
def t_one(values, mu, alternative='two-sided'):
    """
    Проверяет t-тестом одной выборки гипотезу о том, что среднее равно mu.

    Первый инструмент для гипотез этапа 3 о среднем: средний стаж против нормы,
    средний срок рассмотрения против регламента, район против области. Тест —
    `stats.ttest_1samp`; пропуски убираются заранее, иначе scipy вернёт `nan`.
    Альтернатива — как у scipy: `'two-sided'` — среднее не равно mu, `'less'` —
    меньше, `'greater'` — больше; её выбирают до взгляда на данные.

    Аргументы:
        values: Столбец чисел, Series; пропуски допускаются.
        mu: Проверяемое среднее, число.
        alternative: Альтернативная гипотеза, строка 'two-sided', 'less' или
                     'greater'; по умолчанию 'two-sided'.

    Возвращаемое значение:
        Кортеж из двух чисел (t, p): статистика и p-значение stats.ttest_1samp
        по значениям values без пропусков против mu с альтернативой alternative.
        Series не меняется.
    """

    pass

In [ ]:
def t_one_tests():
    values = pd.Series([12.0, 15, 9, None, 20, 14, 18, 11, None, 16], index=list('abcdefghij'))
    before = values.copy()
    result = t_one(values, 12)
    assert isinstance(result, tuple) and len(result) == 2, 'функция должна возвращать кортеж (t, p)'
    t, p = result
    assert round(t, 4) == 1.8344 and round(p, 4) == 0.1092, 't-тест по значениям без пропусков против mu, альтернатива двусторонняя'
    assert round(t_one(values, 16)[0], 4) == -1.2551, 'проверяемое среднее — mu'
    assert round(t_one(values, 12, alternative='greater')[1], 4) == 0.0546, "альтернатива 'greater': p вдвое меньше двустороннего"
    assert round(t_one(values, 12, alternative='less')[1], 4) == 0.9454, "альтернатива 'less'"
    assert values.equals(before), 'Series изменился'

    print('Все тесты прошли успешно!')

In [ ]:
t_one_tests()

In [ ]:
# Проверка: возраст машины водителей области против 10,5 года через t_one, с двусторонней альтернативой и с альтернативой «больше»; печатаются t и p обеих проверок, в тест вводятся t до сотых, оба p до тысячных и решение двусторонней проверки

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

t, p = t_one(drivers_check['vehicle_age'], 10.5)
print(round(t, 2), round(p, 3))
t, p = t_one(drivers_check['vehicle_age'], 10.5, alternative='greater')
print(round(t, 2), round(p, 3))

### Б2. Доверительный интервал для среднего

Интервал — вторая половина ответа о среднем: тест говорит, отличается ли среднее, интервал — насколько. Границы: среднее ± t* · s / √n, где t* — критическое значение распределения Стьюдента с n − 1 степенями свободы, а s / √n — стандартная ошибка `stats.sem`. n — число значений без пропусков: от него зависят и степени свободы, и стандартная ошибка.

Реализуйте функцию `ci_mean(values, conf=0.95)`, которая считает доверительный интервал для среднего по распределению Стьюдента. Аргументы: `values` — столбец чисел, Series; пропуски допускаются; `conf` — уровень доверия, число от 0 до 1; по умолчанию 0.95. Возвращаемое значение: кортеж из двух чисел (low, high): границы stats.t.interval с уровнем доверия conf, n − 1 степенями свободы, центром — средним и масштабом — стандартной ошибкой stats.sem значений values без пропусков; n — число значений без пропусков. Series не меняется.

Подсказка: раздел 2 ноутбука занятия: `values = values.dropna()`; `stats.t.interval(conf, len(values) - 1, loc=values.mean(), scale=stats.sem(values))`. `stats.norm.interval` — нормальное приближение: на маленькой выборке интервал уже нужного; `len` до `dropna()` считает пропуски в степенях свободы; `scale=values.std()` — разброс отдельных значений, а не среднего.

Проверка в ноутбуке: среднее число участников ДТП области через ci_mean с уровнями доверия 0,95 и 0,99; печатаются границы обоих интервалов до тысячных, все четыре вводятся в тест.

In [ ]:
def ci_mean(values, conf=0.95):
    """
    Считает доверительный интервал для среднего по распределению Стьюдента.

    Интервал — вторая половина ответа о среднем: тест говорит, отличается ли
    среднее, интервал — насколько. Границы: среднее ± t* · s / √n, где t* —
    критическое значение распределения Стьюдента с n − 1 степенями свободы, а s
    / √n — стандартная ошибка `stats.sem`. n — число значений без пропусков: от
    него зависят и степени свободы, и стандартная ошибка.

    Аргументы:
        values: Столбец чисел, Series; пропуски допускаются.
        conf: Уровень доверия, число от 0 до 1; по умолчанию 0.95.

    Возвращаемое значение:
        Кортеж из двух чисел (low, high): границы stats.t.interval с уровнем
        доверия conf, n − 1 степенями свободы, центром — средним и масштабом —
        стандартной ошибкой stats.sem значений values без пропусков; n — число
        значений без пропусков. Series не меняется.
    """

    pass

In [ ]:
def ci_mean_tests():
    values = pd.Series([12.0, 15, 9, None, 20, 14, 18, 11, None, 16], index=list('abcdefghij'))
    before = values.copy()
    result = ci_mean(values)
    assert isinstance(result, tuple) and len(result) == 2, 'функция должна возвращать кортеж (low, high)'
    low, high = result
    assert round(low, 4) == 11.3134 and round(high, 4) == 17.4366, 'интервал Стьюдента: восемь значений без пропусков, 7 степеней свободы, масштаб — stats.sem'
    low, high = ci_mean(values, conf=0.9)
    assert round(low, 4) == 11.922 and round(high, 4) == 16.828, 'уровень доверия — conf'
    assert values.equals(before), 'Series изменился'

    print('Все тесты прошли успешно!')

In [ ]:
ci_mean_tests()

In [ ]:
# Проверка: среднее число участников ДТП области через ci_mean с уровнями доверия 0,95 и 0,99; печатаются границы обоих интервалов до тысячных, все четыре вводятся в тест

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])

low, high = ci_mean(df_check['participants_count'])
print(round(low, 3), round(high, 3))
low, high = ci_mean(df_check['participants_count'], conf=0.99)
print(round(low, 3), round(high, 3))

### Б3. Решение словами

Решение в отчёте — слова, а не число: «отвергаем», если p меньше уровня значимости alpha, иначе «не отвергаем»; p ровно alpha — «не отвергаем». p-значение — вероятность, и вне отрезка от 0 до 1 его не бывает, а `nan` — признак пропусков в данных. В обоих случаях функция поднимает ValueError, а не пишет решение: молчаливое «не отвергаем» при `nan` — ловушка тестов scipy.

Реализуйте функцию `decision(p, alpha=0.05)`, которая записывает решение теста словами по p-значению. Аргументы: `p` — p-значение, число; `alpha` — уровень значимости, число; по умолчанию 0.05. Возвращаемое значение: строка «отвергаем», если p < alpha, иначе «не отвергаем». Если p меньше 0, больше 1 или nan — ValueError.

Подсказка: условие `if not 0 <= p <= 1: raise ValueError(...)` в начале функции ловит и `nan`: любое сравнение с `nan` — `False`; затем `'отвергаем' if p < alpha else 'не отвергаем'`. Проверка `if p < 0 or p > 1:` пропускает `nan`.

Проверка в ноутбуке: средний час ДТП области против 13,9 через stats.ttest_1samp, решение через decision при уровнях 0,05 и 0,01; печатаются p до тысячных и оба решения, всё вводится в тест.

In [ ]:
def decision(p, alpha=0.05):
    """
    Записывает решение теста словами по p-значению.

    Решение в отчёте — слова, а не число: «отвергаем», если p меньше уровня
    значимости alpha, иначе «не отвергаем»; p ровно alpha — «не отвергаем».
    p-значение — вероятность, и вне отрезка от 0 до 1 его не бывает, а `nan` —
    признак пропусков в данных. В обоих случаях функция поднимает ValueError, а
    не пишет решение: молчаливое «не отвергаем» при `nan` — ловушка тестов
    scipy.

    Аргументы:
        p: p-значение, число.
        alpha: Уровень значимости, число; по умолчанию 0.05.

    Возвращаемое значение:
        Строка «отвергаем», если p < alpha, иначе «не отвергаем». Если p меньше
        0, больше 1 или nan — ValueError.
    """

    pass

In [ ]:
def decision_tests():
    assert decision(0.01) == 'отвергаем', 'p меньше 0,05 — «отвергаем»'
    assert decision(0.2) == 'не отвергаем', 'p больше 0,05 — «не отвергаем»'
    assert decision(0.05) == 'не отвергаем', 'p ровно alpha — «не отвергаем»: отвергаем только при p < alpha'
    assert decision(0.0) == 'отвергаем' and decision(1.0) == 'не отвергаем', 'границы 0 и 1 допустимы'
    assert decision(0.03, alpha=0.01) == 'не отвергаем' and decision(0.03, alpha=0.1) == 'отвергаем', 'уровень значимости — alpha'
    assert decision(np.float64(1e-20)) == 'отвергаем', 'p — и число NumPy'
    for bad in [1.5, -0.1, float('nan')]:
        try:
            decision(bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f'p = {bad} должно давать ValueError')

    print('Все тесты прошли успешно!')

In [ ]:
decision_tests()

In [ ]:
# Проверка: средний час ДТП области против 13,9 через stats.ttest_1samp, решение через decision при уровнях 0,05 и 0,01; печатаются p до тысячных и оба решения, всё вводится в тест

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])

p = stats.ttest_1samp(df_check['datetime'].dt.hour, 13.9).pvalue
print(round(p, 3), decision(p), decision(p, alpha=0.01))

### Б4. Знаковый тест для медианы

Тест медианы для скошенных данных и выбросов. Если медиана равна m0, значение с равной вероятностью оказывается выше или ниже m0, как монета ложится орлом или решкой. Пропуски и значения, равные m0, отбрасываются, а число значений выше m0 среди остальных сравнивается с половиной биномиальным тестом `stats.binomtest(k, n)`: k — сколько значений выше m0, n — сколько значений не равно m0. Величина отклонений тест не интересует, только знак, поэтому длинный хвост на него не влияет. Знаковый ранговый тест Вилкоксона здесь не годится: он проверяет симметрию, а не медиану, и у скошенного столбца отвергает даже его собственную медиану.

Реализуйте функцию `sign_test(values, m0)`, которая проверяет знаковым тестом гипотезу о том, что медиана равна m0. Аргументы: `values` — столбец чисел, Series; пропуски допускаются; `m0` — проверяемая медиана, число. Возвращаемое значение: кортеж из трёх чисел (k, n, p): k — сколько значений values без пропусков больше m0, целое число; n — сколько значений без пропусков не равно m0, целое число; p — p-значение stats.binomtest(k, n) с параметрами по умолчанию. Series не меняется.

Подсказка: раздел 4 ноутбука занятия: `values = values.dropna()`, `k = int((values > m0).sum())`, `n = int((values != m0).sum())`, `p = stats.binomtest(k, n).pvalue`. Без `dropna()` пропуск считается «не равным m0» и попадает в n; значения, равные m0, в n не входят; `stats.wilcoxon(values - m0)` — другой тест, о симметрии; t-тест проверяет среднее, а не медиану.

Проверка в ноутбуке: возраст машины водителей Воскресенского района против медианы 10 лет через sign_test и для сравнения t-тестом; печатаются k, n и p знакового теста и p t-теста, в тест вводятся k, n, оба p до тысячных и оба решения.

In [ ]:
def sign_test(values, m0):
    """
    Проверяет знаковым тестом гипотезу о том, что медиана равна m0.

    Тест медианы для скошенных данных и выбросов. Если медиана равна m0,
    значение с равной вероятностью оказывается выше или ниже m0, как монета
    ложится орлом или решкой. Пропуски и значения, равные m0, отбрасываются, а
    число значений выше m0 среди остальных сравнивается с половиной биномиальным
    тестом `stats.binomtest(k, n)`: k — сколько значений выше m0, n — сколько
    значений не равно m0. Величина отклонений тест не интересует, только знак,
    поэтому длинный хвост на него не влияет. Знаковый ранговый тест Вилкоксона
    здесь не годится: он проверяет симметрию, а не медиану, и у скошенного
    столбца отвергает даже его собственную медиану.

    Аргументы:
        values: Столбец чисел, Series; пропуски допускаются.
        m0: Проверяемая медиана, число.

    Возвращаемое значение:
        Кортеж из трёх чисел (k, n, p): k — сколько значений values без
        пропусков больше m0, целое число; n — сколько значений без пропусков не
        равно m0, целое число; p — p-значение stats.binomtest(k, n) с
        параметрами по умолчанию. Series не меняется.
    """

    pass

In [ ]:
def sign_test_tests():
    values = pd.Series([3.0, 5, 8, 9, 12, None, 14, 20, 25, 7, 6, None])
    before = values.copy()
    result = sign_test(values, 10)
    assert isinstance(result, tuple) and len(result) == 3, 'функция должна возвращать кортеж (k, n, p)'
    k, n, p = result
    assert isinstance(k, (int, np.integer)) and isinstance(n, (int, np.integer)), 'k и n — целые числа'
    assert (k, n) == (4, 10) and round(p, 4) == 0.7539, 'выше 10 четыре значения из десяти без пропусков'
    k, n, p = sign_test(values, 4)
    assert (k, n) == (9, 10) and round(p, 4) == 0.0215, 'проверяемая медиана — m0'
    k, n, p = sign_test(values, 9)
    assert (k, n) == (4, 9) and round(p, 4) == 1.0, 'значение, равное m0, в n не входит'
    assert values.equals(before), 'Series изменился'

    print('Все тесты прошли успешно!')

In [ ]:
sign_test_tests()

In [ ]:
# Проверка: возраст машины водителей Воскресенского района против медианы 10 лет через sign_test и для сравнения t-тестом; печатаются k, n и p знакового теста и p t-теста, в тест вводятся k, n, оба p до тысячных и оба решения

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

age = drivers_check.loc[drivers_check['district'] == 'Воскресенский', 'vehicle_age']
k, n, p = sign_test(age, 10)
print(k, n, round(p, 3))
print(round(stats.ttest_1samp(age.dropna(), 10).pvalue, 3))

### Б5. z-тест для доли

Доля — тоже среднее, только логического столбца: доля ДТП с погибшими, доля обжалованных решений, доля удовлетворённых исков. Тест — `proportions_ztest` из statsmodels: сначала число успехов, потом число наблюдений, потом проверяемая доля. Стандартная ошибка считается по проверяемой доле, `prop_var=p0`: так тест согласован с интервалом Вилсона из Б6 — отвергает ровно те доли, что лежат вне интервала. Альтернатива называется по-своему, не как у scipy: `'two-sided'` — не равна, `'smaller'` — меньше, `'larger'` — больше.

Реализуйте функцию `share_test(successes, n, p0, alternative='two-sided')`, которая проверяет z-тестом для доли гипотезу о том, что доля успехов равна p0. Аргументы: `successes` — число успехов, целое число; `n` — число наблюдений, целое число; `p0` — проверяемая доля, число от 0 до 1; `alternative` — альтернативная гипотеза, строка 'two-sided', 'smaller' или 'larger'; по умолчанию 'two-sided'. Возвращаемое значение: кортеж из двух чисел (z, p): статистика и p-значение proportions_ztest(successes, n, p0) с альтернативой alternative и prop_var=p0 — стандартной ошибкой по проверяемой доле.

Подсказка: раздел 5 ноутбука занятия: `z, p = proportions_ztest(successes, n, p0, alternative=alternative, prop_var=p0)`. Без `prop_var` statsmodels считает стандартную ошибку по доле выборки — это тест Вальда и другие числа; аргументы в обратном порядке дают огромную z и p 0 без ошибки; scipy-названия `'less'` и `'greater'` statsmodels не принимает: `ValueError`; односторонний p — половина двустороннего, только если доля отклонилась в заявленную сторону, а иначе он близок к единице.

Проверка в ноутбуке: доля наездов на пешехода среди ДТП области против 0,3 через share_test, с двусторонней альтернативой и с альтернативой «больше»; печатаются z и p обеих проверок, в тест вводятся z до сотых, оба p до тысячных и решение односторонней проверки.

In [ ]:
def share_test(successes, n, p0, alternative='two-sided'):
    """
    Проверяет z-тестом для доли гипотезу о том, что доля успехов равна p0.

    Доля — тоже среднее, только логического столбца: доля ДТП с погибшими, доля
    обжалованных решений, доля удовлетворённых исков. Тест — `proportions_ztest`
    из statsmodels: сначала число успехов, потом число наблюдений, потом
    проверяемая доля. Стандартная ошибка считается по проверяемой доле,
    `prop_var=p0`: так тест согласован с интервалом Вилсона из Б6 — отвергает
    ровно те доли, что лежат вне интервала. Альтернатива называется по-своему,
    не как у scipy: `'two-sided'` — не равна, `'smaller'` — меньше, `'larger'` —
    больше.

    Аргументы:
        successes: Число успехов, целое число.
        n: Число наблюдений, целое число.
        p0: Проверяемая доля, число от 0 до 1.
        alternative: Альтернативная гипотеза, строка 'two-sided', 'smaller' или
                     'larger'; по умолчанию 'two-sided'.

    Возвращаемое значение:
        Кортеж из двух чисел (z, p): статистика и p-значение
        proportions_ztest(successes, n, p0) с альтернативой alternative и
        prop_var=p0 — стандартной ошибкой по проверяемой доле.
    """

    pass

In [ ]:
def share_test_tests():
    result = share_test(30, 100, 0.2)
    assert isinstance(result, tuple) and len(result) == 2, 'функция должна возвращать кортеж (z, p)'
    z, p = result
    assert round(z, 4) == 2.5 and round(p, 4) == 0.0124, 'z-тест для доли: 30 успехов из 100 против 0,2, стандартная ошибка по проверяемой доле, prop_var=p0'
    assert round(share_test(30, 100, 0.2, alternative='larger')[1], 4) == 0.0062, "альтернатива 'larger' — доля больше p0"
    assert round(share_test(30, 100, 0.2, alternative='smaller')[1], 4) == 0.9938, "альтернатива 'smaller', а доля выше p0: p близко к единице, а не половина двустороннего"
    z, p = share_test(12, 80, 0.25)
    assert round(z, 4) == -2.0656 and round(p, 4) == 0.0389, 'успехи — successes, наблюдения — n, проверяемая доля — p0'
    assert round(share_test(12, 80, 0.25, alternative='smaller')[1], 4) == 0.0194, "альтернатива 'smaller' — доля меньше p0"
    assert round(share_test(12, 80, 0.25, alternative='larger')[1], 4) == 0.9806, "альтернатива 'larger', а доля ниже p0: p близко к единице"

    print('Все тесты прошли успешно!')

In [ ]:
share_test_tests()

In [ ]:
# Проверка: доля наездов на пешехода среди ДТП области против 0,3 через share_test, с двусторонней альтернативой и с альтернативой «больше»; печатаются z и p обеих проверок, в тест вводятся z до сотых, оба p до тысячных и решение односторонней проверки

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])

hits = df_check['category'] == 'Наезд на пешехода'
z, p = share_test(hits.sum(), len(df_check), 0.3)
print(round(z, 2), round(p, 3))
z, p = share_test(hits.sum(), len(df_check), 0.3, alternative='larger')
print(round(z, 2), round(p, 3))

### Б6. Интервал Вилсона для доли

`proportion_confint` по умолчанию строит интервал по нормальному приближению: доля ± z · √(доля · (1 − доля) / n). На малых выборках и долях у нуля или единицы он ошибается: граница упирается в ноль, а при нуле успехов интервал схлопывается в точку. Интервал Вилсона, `method='wilson'`, таких провалов не даёт. Уровень доверия statsmodels задаёт не им самим, а его дополнением: `alpha = 1 - conf`.

Реализуйте функцию `ci_share(successes, n, conf=0.95)`, которая считает доверительный интервал Вилсона для доли. Аргументы: `successes` — число успехов, целое число; `n` — число наблюдений, целое число; `conf` — уровень доверия, число от 0 до 1; по умолчанию 0.95. Возвращаемое значение: кортеж из двух чисел (low, high): границы интервала proportion_confint(successes, n, alpha=1 − conf, method='wilson').

Подсказка: раздел 5 ноутбука занятия: `proportion_confint(successes, n, alpha=1 - conf, method='wilson')`. Без `method` получится нормальное приближение; `alpha=conf` вместо `1 - conf` даёт узкий интервал уровня 5 %; `method='beta'` — точный интервал Клоппера–Пирсона, шире Вилсона.

Проверка в ноутбуке: доля ДТП с погибшими в Лотошинском районе через ci_share; печатаются число ДТП, число ДТП с погибшими и границы интервала до тысячных, границы вводятся в тест.

In [ ]:
def ci_share(successes, n, conf=0.95):
    """
    Считает доверительный интервал Вилсона для доли.

    `proportion_confint` по умолчанию строит интервал по нормальному
    приближению: доля ± z · √(доля · (1 − доля) / n). На малых выборках и долях
    у нуля или единицы он ошибается: граница упирается в ноль, а при нуле
    успехов интервал схлопывается в точку. Интервал Вилсона, `method='wilson'`,
    таких провалов не даёт. Уровень доверия statsmodels задаёт не им самим, а
    его дополнением: `alpha = 1 - conf`.

    Аргументы:
        successes: Число успехов, целое число.
        n: Число наблюдений, целое число.
        conf: Уровень доверия, число от 0 до 1; по умолчанию 0.95.

    Возвращаемое значение:
        Кортеж из двух чисел (low, high): границы интервала
        proportion_confint(successes, n, alpha=1 − conf, method='wilson').
    """

    pass

In [ ]:
def ci_share_tests():
    result = ci_share(3, 10)
    assert isinstance(result, tuple) and len(result) == 2, 'функция должна возвращать кортеж (low, high)'
    low, high = result
    assert round(low, 4) == 0.1078 and round(high, 4) == 0.6032, "интервал Вилсона, method='wilson': нормальное приближение дало бы 0,016 и 0,584"
    low, high = ci_share(0, 20)
    assert round(low, 4) == 0 and round(high, 4) == 0.1611, 'ноль успехов: интервал Вилсона не схлопывается в точку'
    low, high = ci_share(30, 100, conf=0.9)
    assert round(low, 4) == 0.2307 and round(high, 4) == 0.3798, 'уровень доверия conf: alpha = 1 - conf'

    print('Все тесты прошли успешно!')

In [ ]:
ci_share_tests()

In [ ]:
# Проверка: доля ДТП с погибшими в Лотошинском районе через ci_share; печатаются число ДТП, число ДТП с погибшими и границы интервала до тысячных, границы вводятся в тест

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])

town = df_check[df_check['district'] == 'Лотошинский']
low, high = ci_share((town['dead_count'] > 0).sum(), len(town))
print(len(town), (town['dead_count'] > 0).sum(), round(low, 3), round(high, 3))

### Б7. Хи-квадрат согласия с равномерным распределением

Поровну ли ДТП по дням недели, дела по месяцам, жалобы по судам? Нулевая гипотеза — распределение равномерное: на каждую категорию приходится одна и та же доля наблюдений. Ожидаемые задаются в тех же единицах, что наблюдаемые, — числами, а не долями: общее число, делённое на число категорий; иначе суммы не сойдутся, и scipy поднимет ValueError.

Реализуйте функцию `gof_uniform(counts)`, которая проверяет хи-квадратом согласия, поровну ли наблюдения по категориям. Аргументы: `counts` — счётчики по категориям, Series целых чисел. Возвращаемое значение: кортеж из двух чисел (χ², p): статистика и p-значение stats.chisquare(counts, ожидаемые), где ожидаемые — список из len(counts) одинаковых чисел counts.sum() / len(counts). Series не меняется.

Подсказка: раздел 6 ноутбука занятия: `expected = [counts.sum() / len(counts)] * len(counts)`, `result = stats.chisquare(counts, expected)`. Доли `[1 / len(counts)] * len(counts)` вместо счётчиков дают `ValueError`; `ddof=1` уменьшает число степеней свободы и меняет p; число категорий 7, записанное числом, ломает функцию на других данных.

Проверка в ноутбуке: ДТП области по годам через gof_uniform; печатаются счётчики, χ² до сотых и p до тысячных, в тест вводятся χ², p и решение.

In [ ]:
def gof_uniform(counts):
    """
    Проверяет хи-квадратом согласия, поровну ли наблюдения по категориям.

    Поровну ли ДТП по дням недели, дела по месяцам, жалобы по судам? Нулевая
    гипотеза — распределение равномерное: на каждую категорию приходится одна и
    та же доля наблюдений. Ожидаемые задаются в тех же единицах, что
    наблюдаемые, — числами, а не долями: общее число, делённое на число
    категорий; иначе суммы не сойдутся, и scipy поднимет ValueError.

    Аргументы:
        counts: Счётчики по категориям, Series целых чисел.

    Возвращаемое значение:
        Кортеж из двух чисел (χ², p): статистика и p-значение
        stats.chisquare(counts, ожидаемые), где ожидаемые — список из
        len(counts) одинаковых чисел counts.sum() / len(counts). Series не
        меняется.
    """

    pass

In [ ]:
def gof_uniform_tests():
    counts = pd.Series([12, 8, 10, 15, 5], index=list('abcde'))
    before = counts.copy()
    result = gof_uniform(counts)
    assert isinstance(result, tuple) and len(result) == 2, 'функция должна возвращать кортеж (χ², p)'
    chi2, p = result
    assert round(chi2, 4) == 5.8 and round(p, 4) == 0.2146, 'хи-квадрат против равномерного: ожидаемые по 10 — сумма 50 на пять категорий, 4 степени свободы'
    chi2, p = gof_uniform(pd.Series([30, 10]))
    assert round(chi2, 4) == 10 and round(p, 4) == 0.0016, 'две категории: ожидаемые по 20'
    assert counts.equals(before), 'Series изменился'

    print('Все тесты прошли успешно!')

In [ ]:
gof_uniform_tests()

In [ ]:
# Проверка: ДТП области по годам через gof_uniform; печатаются счётчики, χ² до сотых и p до тысячных, в тест вводятся χ², p и решение

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])

by_year = df_check['datetime'].dt.year.value_counts().sort_index()
chi2, p = gof_uniform(by_year)
print(by_year.tolist(), round(chi2, 2), round(p, 3))

### Б8. Поправка Бонферрони

Если нулевая гипотеза верна, тест ошибается в одном случае из двадцати; на двадцати проверках одна случайная «значимость» — норма. Поправка Бонферрони сравнивает каждое p не с alpha, а с alpha / m, где m — число проверок: тогда вероятность хотя бы одной ложной значимости на всё семейство проверок не больше alpha. Значимо — p меньше порога, как в решении словами.

Реализуйте функцию `bonferroni(pvalues, alpha=0.05)`, которая отмечает проверки, значимые после поправки Бонферрони. Аргументы: `pvalues` — p-значения проверок, Series чисел с метками проверок; `alpha` — уровень значимости на всё семейство проверок, число; по умолчанию 0.05. Возвращаемое значение: объект Series логических значений с тем же индексом, что у pvalues: True, если p < alpha / len(pvalues), иначе False. Series pvalues не меняется.

Подсказка: раздел 7 ноутбука занятия: одна строка — `pvalues < alpha / len(pvalues)`; сравнение Series с числом сохраняет индекс. `alpha * len(pvalues)` — не поправка, а ослабление; поправка Холма, которая проверяет p по очереди, отмечает не меньше проверок, чем Бонферрони, и на тестах даёт другой ответ.

Проверка в ноутбуке: доля ДТП с погибшими по видам ДТП с 50 и более случаями против доли области, p через proportions_ztest с prop_var, поправка через bonferroni; печатаются число видов, число значимых без поправки и с поправкой и виды, потерявшие значимость, в тест вводятся оба числа значимых и вид ДТП.

In [ ]:
def bonferroni(pvalues, alpha=0.05):
    """
    Отмечает проверки, значимые после поправки Бонферрони.

    Если нулевая гипотеза верна, тест ошибается в одном случае из двадцати; на
    двадцати проверках одна случайная «значимость» — норма. Поправка Бонферрони
    сравнивает каждое p не с alpha, а с alpha / m, где m — число проверок: тогда
    вероятность хотя бы одной ложной значимости на всё семейство проверок не
    больше alpha. Значимо — p меньше порога, как в решении словами.

    Аргументы:
        pvalues: p-значения проверок, Series чисел с метками проверок.
        alpha: Уровень значимости на всё семейство проверок, число; по умолчанию
               0.05.

    Возвращаемое значение:
        Объект Series логических значений с тем же индексом, что у pvalues:
        True, если p < alpha / len(pvalues), иначе False. Series pvalues не
        меняется.
    """

    pass

In [ ]:
def bonferroni_tests():
    pvalues = pd.Series([0.001, 0.02, 0.015, 0.3], index=['a', 'b', 'c', 'd'])
    before = pvalues.copy()
    result = bonferroni(pvalues)
    assert isinstance(result, pd.Series), 'функция должна возвращать Series'
    assert result.index.tolist() == ['a', 'b', 'c', 'd'], 'индекс тот же, что у pvalues'
    assert result.dtype == bool, 'значения логические'
    assert result.tolist() == [True, False, False, False], 'порог 0,05 / 4 = 0,0125: 0,015 и 0,02 его не проходят'
    assert bonferroni(pvalues, alpha=0.1).tolist() == [True, True, True, False], 'уровень значимости alpha: порог 0,1 / 4 = 0,025'
    assert bonferroni(pd.Series([0.04], index=['x'])).tolist() == [True], 'одна проверка — порог сам alpha'
    assert pvalues.equals(before), 'Series изменился'

    print('Все тесты прошли успешно!')

In [ ]:
bonferroni_tests()

In [ ]:
# Проверка: доля ДТП с погибшими по видам ДТП с 50 и более случаями против доли области, p через proportions_ztest с prop_var, поправка через bonferroni; печатаются число видов, число значимых без поправки и с поправкой и виды, потерявшие значимость, в тест вводятся оба числа значимых и вид ДТП

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])

dead = df_check['dead_count'] > 0
groups = df_check.assign(dead=dead).groupby('category')['dead'].agg(['sum', 'count'])
groups = groups[groups['count'] >= 50]
p_by_category = pd.Series([proportions_ztest(k, n, dead.mean(), prop_var=dead.mean())[1] for k, n in zip(groups['sum'], groups['count'])], index=groups.index)
significant = bonferroni(p_by_category)
print(len(p_by_category), (p_by_category < 0.05).sum(), significant.sum())
print(p_by_category[(p_by_category < 0.05) & ~significant].round(4))

## Часть В. Задачи повышенной сложности

### В1. Таблица гипотез для отчёта

Сводная таблица раздела гипотез этапа 3: строка на гипотезу, решение словами. Каждая гипотеза — словарь с ключами «вопрос», «тест», «n», «статистика», «p»; другие ключи, например интервал, в сводную таблицу не идут — они для подробной записи. Решение функция добавляет сама через decision из Б3, поэтому p вне отрезка от 0 до 1 или `nan` остановит сборку ошибкой, а не попадёт в отчёт словами «не отвергаем». Словари списка не меняются.

Реализуйте функцию `hypothesis_table(results)`, которая собирает сводную таблицу гипотез для отчёта из списка словарей. Аргументы: `results` — список словарей с ключами «вопрос», «тест», «n», «статистика», «p»; другие ключи допускаются. Возвращаемое значение: объект DataFrame: строка на словарь в порядке списка, индекс 1, 2, 3, …; столбцы «вопрос», «тест», «n», «статистика», «p» — значения из словарей — и «решение» — decision(p) при уровне 0,05. Если p вне отрезка от 0 до 1 или nan — ValueError. Список и словари не меняются.

Подсказка: `columns = ['вопрос', 'тест', 'n', 'статистика', 'p']`; строки — `[{key: row[key] for key in columns} for row in results]`; `pd.DataFrame(строки, index=range(1, len(results) + 1))`; решение — `[decision(p) for p in table['p']]`. `row['решение'] = ...` внутри цикла меняет словари вызывающего; `pd.DataFrame(results)` берёт и лишние ключи.

Проверка в ноутбуке: три гипотезы об области через hypothesis_table — средний возраст машины против 10 лет, доля ДТП в тёмное время суток против 0,38, ДТП по кварталам поровну; печатается таблица, в тест вводятся статистики первых двух строк до сотых, p второй строки до тысячных и решения всех трёх.

In [ ]:
def hypothesis_table(results):
    """
    Собирает сводную таблицу гипотез для отчёта из списка словарей.

    Сводная таблица раздела гипотез этапа 3: строка на гипотезу, решение
    словами. Каждая гипотеза — словарь с ключами «вопрос», «тест», «n»,
    «статистика», «p»; другие ключи, например интервал, в сводную таблицу не
    идут — они для подробной записи. Решение функция добавляет сама через
    decision из Б3, поэтому p вне отрезка от 0 до 1 или `nan` остановит сборку
    ошибкой, а не попадёт в отчёт словами «не отвергаем». Словари списка не
    меняются.

    Аргументы:
        results: Список словарей с ключами «вопрос», «тест», «n», «статистика»,
                 «p»; другие ключи допускаются.

    Возвращаемое значение:
        Объект DataFrame: строка на словарь в порядке списка, индекс 1, 2, 3, …;
        столбцы «вопрос», «тест», «n», «статистика», «p» — значения из словарей
        — и «решение» — decision(p) при уровне 0,05. Если p вне отрезка от 0 до
        1 или nan — ValueError. Список и словари не меняются.
    """

    pass

In [ ]:
def hypothesis_table_tests():
    results = [
        {'вопрос': 'средний срок больше 10 дней?', 'тест': 't-тест', 'n': 8, 'статистика': 2.1, 'p': 0.07},
        {'вопрос': 'доля обжалованных 0,2?', 'тест': 'z-тест доли', 'n': 100, 'статистика': 2.18, 'p': 0.029, 'интервал': (0.22, 0.4)},
        {'вопрос': 'дела по дням поровну?', 'тест': 'хи-квадрат', 'n': 50, 'статистика': 5.8, 'p': 0.05},
    ]
    before = [dict(row) for row in results]
    table = hypothesis_table(results)
    assert isinstance(table, pd.DataFrame), 'функция должна возвращать DataFrame'
    assert table.columns.tolist() == ['вопрос', 'тест', 'n', 'статистика', 'p', 'решение'], 'столбцы как в условии, в том же порядке; лишние ключи словарей в таблицу не попадают'
    assert table.index.tolist() == [1, 2, 3], 'строки нумеруются с единицы в порядке списка'
    assert table['вопрос'].tolist() == ['средний срок больше 10 дней?', 'доля обжалованных 0,2?', 'дела по дням поровну?'], 'значения из словарей'
    assert table['n'].tolist() == [8, 100, 50] and table['p'].tolist() == [0.07, 0.029, 0.05], 'значения из словарей'
    assert table['решение'].tolist() == ['не отвергаем', 'отвергаем', 'не отвергаем'], 'решение — decision(p) при 0,05; p ровно 0,05 — «не отвергаем»'
    assert results == before, 'список словарей изменился'
    try:
        hypothesis_table([{'вопрос': 'срок?', 'тест': 't-тест', 'n': 0, 'статистика': float('nan'), 'p': float('nan')}])
    except ValueError:
        pass
    else:
        raise AssertionError('p = nan должно давать ValueError, как у decision')

    print('Все тесты прошли успешно!')

In [ ]:
hypothesis_table_tests()

In [ ]:
# Проверка: три гипотезы об области через hypothesis_table — средний возраст машины против 10 лет, доля ДТП в тёмное время суток против 0,38, ДТП по кварталам поровну; печатается таблица, в тест вводятся статистики первых двух строк до сотых, p второй строки до тысячных и решения всех трёх

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

age = drivers_check['vehicle_age']
dark = df_check['light'].str.contains('темное')
quarters = df_check['datetime'].dt.quarter.value_counts().sort_index()
t, p_t = t_one(age, 10)
z, p_z = share_test(dark.sum(), len(dark), 0.38)
chi2, p_chi = gof_uniform(quarters)
hypotheses = [
    {'вопрос': 'средний возраст машины 10 лет?', 'тест': 't-тест', 'n': age.count(), 'статистика': t, 'p': p_t},
    {'вопрос': 'доля ДТП в тёмное время 0,38?', 'тест': 'z-тест доли', 'n': len(dark), 'статистика': z, 'p': p_z},
    {'вопрос': 'ДТП по кварталам поровну?', 'тест': 'хи-квадрат', 'n': quarters.sum(), 'статистика': chi2, 'p': p_chi},
]
print(hypothesis_table(hypotheses).round(3))

### В2. Проверки по районам с поправкой

Район против области, сразу по всем районам: строка на район, p и два решения — без поправки и с поправкой. Проверяются только районы, где значений без пропусков не меньше min_n: на десятке значений тест слаб, а районов много. Поправка Бонферрони делит alpha на число проверенных районов, а не всех. Функция пользуется t_one из Б1 и bonferroni из Б8.

Реализуйте функцию `district_tests(df, column, mu, min_n=50, alpha=0.05)`, которая проверяет t-тестом среднее столбца в каждом районе против mu, с поправкой Бонферрони на число районов. Аргументы: `df` — таблица со столбцом district и числовым столбцом column; `column` — имя числового столбца, строка; `mu` — проверяемое среднее, число; `min_n` — наименьшее число значений без пропусков в районе, целое число; по умолчанию 50; `alpha` — уровень значимости, число; по умолчанию 0.05. Возвращаемое значение: объект DataFrame: строка на каждый район с не меньше чем min_n значениями column без пропусков, районы по алфавиту, как их упорядочивает groupby; столбцы «n» — число значений без пропусков, «среднее», «p» — p двустороннего t_one против mu, «значимо» — p < alpha, «значимо после поправки» — bonferroni по столбцу p с тем же alpha. Таблица df не меняется.

Подсказка: цикл `for name, values in df.groupby('district')[column]:` — `values = values.dropna()`, отбор `len(values) >= min_n`, строки собираются в словарь `{name: {'n': ..., 'среднее': ..., 'p': ...}}`; таблица — `pd.DataFrame.from_dict(rows, orient='index')`; затем `table['p'] < alpha` и `bonferroni(table['p'], alpha)`. Отбор по числу строк считает пропуски; поправка на число всех районов строже нужной.

Проверка в ноутбуке: среднее число участников ДТП по районам области, где 250 и больше ДТП, против среднего области через district_tests; печатаются число районов, число значимых без поправки и с поправкой и три района с наименьшим p, в тест вводятся три числа и район с наименьшим p.

In [ ]:
def district_tests(df, column, mu, min_n=50, alpha=0.05):
    """
    Проверяет t-тестом среднее столбца в каждом районе против mu, с поправкой
    Бонферрони на число районов.

    Район против области, сразу по всем районам: строка на район, p и два
    решения — без поправки и с поправкой. Проверяются только районы, где
    значений без пропусков не меньше min_n: на десятке значений тест слаб, а
    районов много. Поправка Бонферрони делит alpha на число проверенных районов,
    а не всех. Функция пользуется t_one из Б1 и bonferroni из Б8.

    Аргументы:
        df: Таблица со столбцом district и числовым столбцом column.
        column: Имя числового столбца, строка.
        mu: Проверяемое среднее, число.
        min_n: Наименьшее число значений без пропусков в районе, целое число; по
               умолчанию 50.
        alpha: Уровень значимости, число; по умолчанию 0.05.

    Возвращаемое значение:
        Объект DataFrame: строка на каждый район с не меньше чем min_n
        значениями column без пропусков, районы по алфавиту, как их
        упорядочивает groupby; столбцы «n» — число значений без пропусков,
        «среднее», «p» — p двустороннего t_one против mu, «значимо» — p < alpha,
        «значимо после поправки» — bonferroni по столбцу p с тем же alpha.
        Таблица df не меняется.
    """

    pass

In [ ]:
def district_tests_tests():
    df = pd.DataFrame({
        'district': ['Б', 'А', 'А', 'В', 'Б', 'А', 'Б', 'В', 'А', 'Б', 'Г', 'В'],
        'x': [7, 11, 12, None, 6, 13, 5, 30, 14, None, 7, 31],
        'name': list('qwertyuiopas'),
    })
    before = df.copy()
    table = district_tests(df, 'x', 10, min_n=3)
    assert isinstance(table, pd.DataFrame), 'функция должна возвращать DataFrame'
    assert table.columns.tolist() == ['n', 'среднее', 'p', 'значимо', 'значимо после поправки'], 'столбцы как в условии, в том же порядке'
    assert table.index.tolist() == ['А', 'Б'], 'районы по алфавиту; в районе В два значения и пропуск — меньше min_n, в Г одно значение'
    assert table['n'].tolist() == [4, 3], 'n — число значений без пропусков; ровно min_n значений — район проверяется'
    assert table['среднее'].tolist() == [12.5, 6.0], 'среднее значений района'
    assert table['p'].round(4).tolist() == [0.0305, 0.0202], 'p двустороннего t-теста против mu по значениям без пропусков; в районе Б среднее ниже mu, и p — тоже двусторонний'
    assert table['значимо'].tolist() == [True, True], 'значимо — p < alpha'
    assert table['значимо после поправки'].tolist() == [False, True], 'поправка Бонферрони на число проверенных районов, два: порог 0,025'
    assert district_tests(df, 'x', 10, min_n=3, alpha=0.1)['значимо после поправки'].tolist() == [True, True], 'уровень значимости alpha передаётся и в поправку'
    assert df.equals(before), 'таблица изменилась'

    print('Все тесты прошли успешно!')

In [ ]:
district_tests_tests()

In [ ]:
# Проверка: среднее число участников ДТП по районам области, где 250 и больше ДТП, против среднего области через district_tests; печатаются число районов, число значимых без поправки и с поправкой и три района с наименьшим p, в тест вводятся три числа и район с наименьшим p

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])

table = district_tests(df_check, 'participants_count', df_check['participants_count'].mean(), min_n=250)
print(len(table), table['значимо'].sum(), table['значимо после поправки'].sum())
print(table.sort_values('p').head(3).round({'среднее': 2}))

### В3. Объём выборки для интервала

Сколько дел изучить, чтобы узнать средний срок с точностью ±2 дня? Половина ширины интервала — z · std / √n; отсюда n = (z · std / half_width)², округлённое вверх: дробного числа дел не бывает, а округление вниз даст интервал шире заданного. z — квантиль нормального распределения уровня 1 − (1 − conf) / 2: n ещё неизвестно, а при n в десятки и сотни распределение Стьюдента почти не отличается от нормального. std берут из прошлых данных или пробной выборки.

Реализуйте функцию `sample_size_mean(std, half_width, conf=0.95)`, которая считает, сколько значений нужно, чтобы интервал для среднего был не шире заданного. Аргументы: `std` — стандартное отклонение значений, положительное число; `half_width` — половина ширины интервала — допустимая погрешность среднего, положительное число; `conf` — уровень доверия, число от 0 до 1; по умолчанию 0.95. Возвращаемое значение: целое число int: (z · std / half_width)², округлённое вверх до целого, где z = stats.norm.ppf(1 − (1 − conf) / 2).

Подсказка: `z = stats.norm.ppf(1 - (1 - conf) / 2)`; `int(np.ceil((z * std / half_width) ** 2))`: `np.ceil` возвращает дробное число, `int` делает из него целое. `round` округляет и вниз; `stats.norm.ppf(conf)` — квантиль одностороннего интервала; z = 1,96, записанное числом, годится только для 95 %.

Проверка в ноутбуке: объём выборки для среднего возраста машины через sample_size_mean с отклонением по водителям области и погрешностью 0,25 года при уровнях доверия 0,95 и 0,99; печатаются отклонение и оба объёма, объёмы вводятся в тест.

In [ ]:
def sample_size_mean(std, half_width, conf=0.95):
    """
    Считает, сколько значений нужно, чтобы интервал для среднего был не шире
    заданного.

    Сколько дел изучить, чтобы узнать средний срок с точностью ±2 дня? Половина
    ширины интервала — z · std / √n; отсюда n = (z · std / half_width)²,
    округлённое вверх: дробного числа дел не бывает, а округление вниз даст
    интервал шире заданного. z — квантиль нормального распределения уровня 1 −
    (1 − conf) / 2: n ещё неизвестно, а при n в десятки и сотни распределение
    Стьюдента почти не отличается от нормального. std берут из прошлых данных
    или пробной выборки.

    Аргументы:
        std: Стандартное отклонение значений, положительное число.
        half_width: Половина ширины интервала — допустимая погрешность среднего,
                    положительное число.
        conf: Уровень доверия, число от 0 до 1; по умолчанию 0.95.

    Возвращаемое значение:
        Целое число int: (z · std / half_width)², округлённое вверх до целого,
        где z = stats.norm.ppf(1 − (1 − conf) / 2).
    """

    pass

In [ ]:
def sample_size_mean_tests():
    n = sample_size_mean(10, 2)
    assert isinstance(n, (int, np.integer)), 'функция должна возвращать целое число'
    assert n == 97, 'n = (z · std / half_width)² = 96,04 округляется вверх: 97; z — квантиль нормального распределения уровня 0,975'
    assert sample_size_mean(10, 2, conf=0.99) == 166, 'уровень доверия conf: квантиль 0,995'
    assert sample_size_mean(12, 0.5) == 2213, 'half_width — половина ширины интервала'
    assert sample_size_mean(3, 3) == 4, 'при погрешности, равной отклонению, нужно четыре значения'

    print('Все тесты прошли успешно!')

In [ ]:
sample_size_mean_tests()

In [ ]:
# Проверка: объём выборки для среднего возраста машины через sample_size_mean с отклонением по водителям области и погрешностью 0,25 года при уровнях доверия 0,95 и 0,99; печатаются отклонение и оба объёма, объёмы вводятся в тест

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

std = drivers_check['vehicle_age'].std()
print(round(std, 3), sample_size_mean(std, 0.25), sample_size_mean(std, 0.25, conf=0.99))

## Необязательная задача. Первые две гипотезы этапа 3 для своего датасета

Этап 3 проекта выдан сегодня, срок — семинар 14: не меньше четырёх гипотез разных видов на своём датасете, каждая записана по формату из раздела 7 ноутбука занятия — вопрос, H0 и H1, тест и почему, n, статистика и p, интервал или размер эффекта, вывод словами с оговоркой о практической значимости. Первые две гипотезы этапа — об одной выборке, и их можно сделать сейчас; записанные по формату, они переходят в отчёт этапа 3 как есть. Две оставшиеся — сравнение групп и связь категорий — появятся на семинаре 13.

Гипотеза 1 — нормальность главного числового столбца, то есть его согласие с нормальным распределением: тест, решение и что дальше, как в разделе «Нормальность» семинара 11; если необязательная задача семинара 11 сделана, её таблица и Q–Q графики берутся оттуда, а здесь гипотеза записывается по формату. Гипотеза 2 — среднее числового столбца против известного числа, `t_one` и `ci_mean`, или доля против известной доли, `share_test` и `ci_share`. Известное значение берётся из нормы, отчёта, другого периода или из всей таблицы против её части, и откуда оно взято, пишется в вопросе. Альтернатива выбирается до расчёта.

Если в датасете есть категориальный столбец с понятным ожидаемым распределением — дни недели, месяцы, — можно добавить согласие с равномерным распределением, `gof_uniform`: в этапе 3 это пятая гипотеза и бонус.

Если датасета ещё нет, возьмите таблицы Брянской области `datasets/legal/dtp-bryansk.csv` и `datasets/legal/dtp-bryansk-participants.csv` и вопросы, которых нет в ноутбуке занятия: не стаж против 15 лет, не Жуковский район и не доля ДТП с погибшими против 0,15.

В ответ в тест Moodle вставьте код и записи обеих гипотез; вложений у этого вопроса нет. Балл ставит преподаватель; автоматической проверки нет.

In [ ]:
path = '../datasets/legal/dtp-bryansk.csv'   # путь к своему файлу; пока таблица ДТП Брянской области
data = pd.read_csv(path)

print(data.shape)
print(data.dtypes)

In [ ]:
# гипотеза 1: нормальность главного числового столбца

In [ ]:
# гипотеза 2: среднее или доля против известного значения

Записи гипотез

| Пункт | Гипотеза 1 | Гипотеза 2 |
|---|---|---|
| вопрос | | |
| H0 и H1 | | |
| тест и почему | | |
| n | | |
| статистика и p | | |
| интервал или размер эффекта | | |
| вывод | | |